# Model Evaluation Tool

This notebook uses saved trained models to obtain predictions and evaluate performances based on chosen metrics.

In [ ]:
%reset -f

In [ ]:
import sys
sys.path.append('../')
import pyrul
import pandas as pd
import tqdm
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Paths information
model_name = "DSCNN" # <- indicate the name of the chosen model
dataset_name = "cmapss" # <- indicate on which dataset the model is evaluated
data_root = "../best_models/"+model_name+"/"+dataset_name+"/"
data_processed_paths = [data_root+"trial"+str(i)+"/preprocessed_data/"+model_name+"_paper_processed" for i in range(10)]
data_metrics_paths = [data_root+"trial"+str(i)+"/data_metrics/" for i in range(10)]
model_paths = [data_root+"trial"+str(i)+"/"+model_name for i in range(10)]
pred_paths = [data_root+"trial"+str(i)+"/preds/" for i in range(10)]

In [ ]:
# Test data load path based on chosen dataset
if dataset_name == "cmapss":
    data_test_load_path = "../datasets/cmapss/test_FD001_fixed.txt"

elif dataset_name == "femto":
    if model_name in ["LR","SVR","MLS","TSBPDTW","DSCNN","RULRVE","DeepLSTM","DAST"]:
        data_test_load_path = "../datasets/femto_alt/test_data.csv"
    elif model_name in ["GBT","GCUTCN"]:
        data_test_load_path = "../datasets/femto/test_data.csv"
    else:
        raise ValueError("Model name not found.")

print("Path=", data_test_load_path)
tmp_plot_path = "../tmp_plots/"

In [ ]:
# Fixing the seed
SEED = pyrul.utils.seed.get_seed(0)
pyrul.utils.seed.set_seed(SEED)

In [ ]:
# Clearing the temporary plots folder
pyrul.utils.file.clear_tmp_plots("/NAS/PROJECTS/SIM_DOCTORANTS/e093728/rul-thesis/python_code/tmp_plots/")

Using the <b>Evaluator</b> objects based on chosen <b>metrics</b>, one can evaluate saved models on chosen dataset(s). 

In this example, the DSCNN is evaluated on the test and full test datasets, for all seeds (=trials). 

Chosen metrics are:
- "scoring_"+dataset_name : the scoring function (dataset-dependent)
- "rmse" : the root mean squared error
- "mae" : the mean absolute error
- "pmae" : the positive absolute error (over-prediction)
- "sstd" : the sample (unit-wise) standard deviation
- "pearsonr" : the Pearson correlation coefficient
- "it" : the inference time (can be long to compute!)

It is possible to specify which samples to look at with a prefix:
- "phase0_[metric]" means to look only at samples whose RUL values are greater than or equal to the model._R_EARLY threshold
- "phase1_[metric]" means to look only at samples whose RUL values are less

By default, all samples are looked at if there is no threshold.

Further details on metrics and custom metrics can be found in pyrul/evaluation/metrics.py

## I. Model performances

In [ ]:
# Main algorithm
mets = [] # list of obtained performances
for i in tqdm.tqdm(range(10)):

    # Model and Dataloader creation
    model = pyrul.models.dscnn.DSCNN_Model(
            DATASET_NAME = "cmapss_fd001",
            LEARNING_RATE = 6e-5,
            SEED=SEED)

    dl = pyrul.models.data.DataLoader(dataset="cmapss_fd001")
    
    # Loading datasets/data metrics
    #dl.load_datasets(model=model, data=["train","val"], path=data_processed_paths[i])
    dl.load_data_metrics(model=model, path=data_metrics_paths[i])
    dl.build_test_data(model, data_path=data_test_load_path)
    
    # Building core models
    model.build_model()
    #model.fit_model() # should be used for IBL models like MLS and TSBPDTW, instead of load_models()
    model.load_models(model_paths[i])

    # Creating the Evaluator with chosen metrics
    evaluator = pyrul.evaluation.Evaluator(metrics=["scoring_"+dataset_name,
                                                    "rmse",
                                                    "mae",
                                                    "pmae",
                                                    "sstd",
                                                    "it",
                                                    "phase0_rmse",
                                                    "phase1_rmse",
                                                    "phase1_mae",
                                                    "phase1_pmae",
                                                    "phase0_sstd",
                                                    "phase1_sstd",
                                                    "pearsonr"])

    met = []
    # All possible dataset types to be evaluated include : "train","val","test","test_full","test_ind","test_full_ind"
    # "_ind" indicates that each fold is evaluated and kept individually instead of being averaged
    for d in ["test","test_full"]:

        # If predictions need to be saved
        #df_metrics = evaluator.get_metrics(model, data=d, save_pred_path=pred_paths[i])
        
        # If predictions are available to be loaded
        df_metrics = evaluator.get_metrics(model, data=d, load_pred_path=pred_paths[i])

        # If predictions should not be saved
        #df_metrics = evaluator.get_metrics(model, data=d)

        met.append(df_metrics.mean())
    res = pd.concat(met)
    mets.append(res)

final_res = pd.DataFrame(mets)

In [ ]:
# Saving the performances to a CSV file
final_res.to_csv(model_name+"_"+dataset_name+"_metrics.csv")

## II. Visualisation of predictions

In [ ]:
i = 0

model = pyrul.models.dscnn.DSCNN_Model(
        DATASET_NAME = "cmapss_fd001",
        LEARNING_RATE = 6e-5,
        SEED=SEED)

dl = pyrul.models.data.DataLoader(dataset="cmapss_fd001")

dl.load_data_metrics(model=model, path=data_metrics_paths[i])
dl.build_test_data(model, data_path=data_test_load_path)

model.build_model()
model.load_models(model_paths[i])

In [ ]:
pyrul.models.viz.plot_pred(model, 
                            plot_indices=[94,17,52],
                            fold_indices=[-1,-1,-1],
                            model_indices=[-1,-1,-1], 
                            data_name="test_full", 
                            save_path=tmp_plot_path,
                            load_pred_path=["../best_models/DSCNN/cmapss/trial0/preds/test_full"]*3)

## List of all best models

Below are given the codes to quickly instanciate each best trained model for the benchmark.

In [ ]:
"""model = pyrul.models.lr.LR_Model(
        DATASET_NAME = "cmapss_fd001",
        SEED=SEED)"""

In [ ]:
"""model = pyrul.models.svr.SVR_Model(
        DATASET_NAME = "cmapss_fd001",
        WINDOW = 30,
        SG_WINDOW = 31,
        GAMMA = 28.77,
        C = 1000,
        EPSILON = 0.99,
        SEED=SEED)"""

In [ ]:
"""model = pyrul.models.mls.MLS_Model(
        DATASET_NAME = "cmapss_fd001",
        WINDOW=20,
        SLIDING_STEP=10,
        GF_SIGMA=4.13,
        ALPHA=0,
        BETA=252,
        LAMBDA1=542,
        LAMBDA2=953,
        DELTA_W=0,
        K=1,
        SEED=SEED)"""

In [ ]:
"""model = pyrul.models.tsbpdtw.TSBPDTW_Model(
        DATASET_NAME = "cmapss_fd001",
        KR_ALPHA=0.01,
        KR_GAMMA=7.84,
        DTW_TOL=4,
        NB_SIM=10,
        INTER_FACTOR=0.19,
        D1=187,
        D2=1,
        SEED=SEED)"""

In [ ]:
"""model = pyrul.models.dscnn.DSCNN_Model(
        DATASET_NAME = "cmapss_fd001",
        LEARNING_RATE = 6e-5,
        SEED=SEED)"""

In [ ]:
"""model = pyrul.models.deeplstm.DeepLSTM_Model(
        DATASET_NAME = "cmapss_fd001",
        BATCH_SIZE = 128,
        LEARNING_RATE=0.01,
        ES=10,
        EPOCHS=300,
        LSTM1_DP=0,
        LSTM2_DP=0.98,
        FNN1_DP=0,
        FNN2_DP=0,
        L2=40.88,
        SEED=SEED)"""

In [ ]:
"""model = pyrul.models.rulrve.RULRVE_Model(
        DATASET_NAME = "cmapss_fd001",
        SEED=SEED)"""

In [ ]:
"""model = pyrul.models.dast.DAST_Model(
        DATASET_NAME = "cmapss_fd001",
        WINDOW_DEC=15,
        HEADS=3,
        NB_NEURONS_VAL=16,
        NB_ENC=1,
        NB_DEC=1,
        SEED=SEED)"""

In [ ]:
"""model = pyrul.models.gbt.GBT_Model(
        DATASET_NAME = "cmapss_fd001",
        LEARNING_RATE = 0.3,
        N_ESTIMATORS = 10,
        MIN_SAMPLES_SPLIT = 58,
        MIN_SAMPLES_LEAF = 100,
        MAX_DEPTH = 5,
        WINDOW = 5,
        SEED=SEED)"""

In [ ]:
"""model = pyrul.models.gcutcn.GCUTCN_Model(
        DATASET_NAME = "cmapss_fd001",
        NB_FEATURES = 14,
        WINDOW = 16,
        EPOCHS = 30,
        BATCH_SIZE = 256,
        LEARNING_RATE = 0.001,
        SEED=SEED)"""